In [1]:
import json, math
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

old = pd.DataFrame(json.load(open("../data/raw/resort_open_data.json", encoding="utf-8"))["resorts"]).set_index("resort_id")
new = pd.DataFrame(json.load(open("../data/raw/resort_open_data_v2.json", encoding="utf-8"))["resorts"]).set_index("resort_id")
assert list(old.index) == list(new.index), "наборы курортов различаются"

metrics = ["piste_segments_6km", "lift_lines_6km", "lodging_objects_6km", "food_objects_6km",
           "transport_objects_6km", "attraction_objects_6km", "avg_snowfall_cm_per_season",
           "avg_freezing_days_per_season", "avg_snow_days_per_season", "nearest_airport_distance_km"]

# 1. ошибки источников (пустые значения не считаем ошибкой)
err_cols = [c for c in new.columns if c.endswith("_error")]
has_error = new[err_cols].fillna("").astype(str).ne("").any(axis=1) if err_cols else pd.Series(False, index=new.index)
print("Курорты с ошибками источников:", new.index[has_error].tolist())
print("Пропуски в показателях:", new[metrics].isna().sum()[lambda s: s > 0].to_dict())

# 2. Википедия против вашей ручной проверки (независимая сверка координат)
ok_wiki = new["wiki_coordinate_quality"].isin(["resort_or_complex_page", "nearby_feature_proxy"])
wm = pd.Series({i: haversine(new.at[i, "wiki_latitude"], new.at[i, "wiki_longitude"],
                             new.at[i, "latitude"], new.at[i, "longitude"]) for i in new.index[ok_wiki]})
print(f"Википедия дала координату для {len(wm)} курортов; расхождение с ручной точкой > 1.5 км у {(wm > 1.5).sum()}")
print(wm[wm > 1.5].round(1).rename("км").to_frame().join(new["name_ru"]))

# 3. что изменилось против старого сбора
cmp = pd.DataFrame({"name": new["name_ru"]})
cmp["shift_km"] = [haversine(old.at[i, "latitude"], old.at[i, "longitude"],
                             new.at[i, "latitude"], new.at[i, "longitude"]) for i in new.index]
cmp["moved"] = cmp["shift_km"] > 0.05
for m in metrics:
    cmp[m + "_old"], cmp[m + "_new"] = old[m], new[m]
    cmp[m + "_diff"] = new[m] - old[m]

summary = cmp.groupby("moved")[[m + "_diff" for m in metrics]].agg(lambda s: s.abs().median()).T
summary.columns = [("точка сдвинута" if c else "точка не двигалась") for c in summary.columns]
print(cmp["moved"].value_counts().to_dict())
cmp.to_csv("../data/prepared/collect_comparison.csv", index=False, encoding="utf-8-sig")
summary.round(1)

Курорты с ошибками источников: ['R015']
Пропуски в показателях: {}
Википедия дала координату для 26 курортов; расхождение с ручной точкой > 1.5 км у 9
        км         name_ru
R002   2.0  Красная Поляна
R010   2.8             Цей
R012   5.6        Телецкий
R014   5.9    Кукисвумчорр
R020   2.6  Соболиная гора
R023   5.1      Аджигардак
R025   5.6       Завьялиха
R026   2.1        Абзаково
R032  17.4      Гладенькая
{True: 20, False: 12}


,точка не двигалась,точка сдвинута
piste_segments_6km_diff,0.0,0.5
lift_lines_6km_diff,0.0,0.0
lodging_objects_6km_diff,0.0,2.5
food_objects_6km_diff,0.0,1.0
transport_objects_6km_diff,0.0,2.5
attraction_objects_6km_diff,0.0,2.0
avg_snowfall_cm_per_season_diff,22.2,24.3
avg_freezing_days_per_season_diff,4.8,10.1
avg_snow_days_per_season_diff,10.9,8.5
nearest_airport_distance_km_diff,0.0,2.0


In [2]:
for c in err_cols:
    print(c, new[c].dropna().astype(str).value_counts().to_dict())

coordinate_error {'No suitable coordinate page': 1}


In [3]:
check = new[["name_ru", "avg_snowfall_cm_per_season", "avg_snow_days_per_season",
             "avg_freezing_days_per_season", "nearest_airport_distance_km",
             "piste_segments_6km", "lift_lines_6km"]].copy()
check["снежных_дней_раньше"] = old["avg_snow_days_per_season"]
check["снегопад_раньше"] = old["avg_snowfall_cm_per_season"]
check.describe().round(1)

,avg_snowfall_cm_per_season,avg_snow_days_per_season,avg_freezing_days_per_season,nearest_airport_distance_km,piste_segments_6km,lift_lines_6km,снежных_дней_раньше,снегопад_раньше
count,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0
mean,162.5,75.9,128.8,92.2,49.6,13.0,85.8,225.5
std,78.5,25.5,39.4,56.1,76.3,16.4,16.6,137.9
min,36.2,16.8,18.6,9.5,5.0,1.0,55.0,74.2
25%,97.8,62.0,113.4,43.1,12.0,4.0,76.1,117.0
50%,145.3,74.0,138.5,91.6,19.5,8.0,87.3,186.3
75%,211.0,86.6,154.9,131.2,42.2,12.2,100.0,334.1
max,351.8,129.0,182.2,231.3,334.0,64.0,109.4,564.3


In [4]:
cols = ["weather_model", "weather_period_start", "weather_period_end"]
print("старый:", old[cols].drop_duplicates().to_dict("records"))
print("новый: ", new[cols].drop_duplicates().to_dict("records"))

same = cmp.index[~cmp["moved"]]
diag = pd.DataFrame({
    "курорт": new.loc[same, "name_ru"],
    "снегопад_старый": old.loc[same, "avg_snowfall_cm_per_season"].round(0),
    "снегопад_новый": new.loc[same, "avg_snowfall_cm_per_season"].round(0),
    "сн_дни_старый": old.loc[same, "avg_snow_days_per_season"].round(0),
    "сн_дни_новый": new.loc[same, "avg_snow_days_per_season"].round(0),
    "мороз_старый": old.loc[same, "avg_freezing_days_per_season"].round(0),
    "мороз_новый": new.loc[same, "avg_freezing_days_per_season"].round(0),
})
diag["снегопад_новый/старый"] = (diag["снегопад_новый"] / diag["снегопад_старый"]).round(2)
diag

старый: [{'weather_model': 'ERA5', 'weather_period_start': '2021-11-01', 'weather_period_end': '2026-04-30'}]
новый:  [{'weather_model': 'ERA5', 'weather_period_start': '2021-11-01', 'weather_period_end': '2026-04-30'}]


,курорт,снегопад_старый,снегопад_новый,сн_дни_старый,сн_дни_новый,мороз_старый,мороз_новый,снегопад_новый/старый
resort_id,,,,,,,,
R001,Роза Хутор,390.0,120.0,93.0,32.0,35.0,35.0,0.31
R003,Газпром Поляна,390.0,36.0,93.0,17.0,19.0,19.0,0.09
R004,Шерегеш,244.0,267.0,93.0,104.0,152.0,162.0,1.09
R005,Архыз,523.0,261.0,108.0,63.0,82.0,82.0,0.50
R008,Чегет,402.0,294.0,100.0,89.0,135.0,142.0,0.73
R015,КолаСпортЛэнд,186.0,211.0,107.0,122.0,157.0,168.0,1.13
R016,Горный воздух,204.0,201.0,82.0,80.0,151.0,155.0,0.99
R019,Бобровый лог,93.0,71.0,84.0,73.0,140.0,146.0,0.76
R021,Гора Белая,117.0,123.0,75.0,78.0,145.0,155.0,1.05


In [5]:
rows = []
for i in wm[wm > 1.5].index:
    r = new.loc[i]
    rows.append({
        "id": i, "курорт": r["name_ru"],
        "страница_википедии": str(r["wiki_coordinate_source_url"]).rsplit("/", 1)[-1],
        "качество_википедии": r["wiki_coordinate_quality"],
        "вики_от_ручной_км": round(haversine(r["wiki_latitude"], r["wiki_longitude"], r["latitude"], r["longitude"]), 1),
        "вики_от_исходной_км": round(haversine(r["wiki_latitude"], r["wiki_longitude"], r["fallback_lat"], r["fallback_lon"]), 1),
        "ручная_от_исходной_км": round(haversine(r["latitude"], r["longitude"], r["fallback_lat"], r["fallback_lon"]), 1),
    })
disputed = pd.DataFrame(rows).sort_values("вики_от_ручной_км", ascending=False)
disputed.to_csv("../data/prepared/coordinate_disputes.csv", index=False, encoding="utf-8-sig")
disputed

,id,курорт,страница_википедии,качество_википедии,вики_от_ручной_км,вики_от_исходной_км,ручная_от_исходной_км
8,R032,Гладенькая,%D0%A1%D0%B0%D1%8F%D0%BD%D0%BE%D0%B3%D0%BE%D1%...,nearby_feature_proxy,17.4,0.0,17.4
3,R014,Кукисвумчорр,%D0%9A%D0%B8%D1%80%D0%BE%D0%B2%D1%81%D0%BA_(%D...,nearby_feature_proxy,5.9,6.4,0.6
2,R012,Телецкий,%D0%98%D0%BE%D0%B3%D0%B0%D1%87_(%D1%80%D0%B5%D...,nearby_feature_proxy,5.6,1.9,5.2
6,R025,Завьялиха,%D0%A2%D1%80%D1%91%D1%85%D0%B3%D0%BE%D1%80%D0%...,nearby_feature_proxy,5.6,11.0,5.7
5,R023,Аджигардак,%D0%90%D0%B4%D0%B6%D0%B8%D0%B3%D0%B0%D1%80%D0%...,nearby_feature_proxy,5.1,2.2,4.2
1,R010,Цей,%D0%A6%D0%B5%D0%B9,nearby_feature_proxy,2.8,0.0,2.8
4,R020,Соболиная гора,%D0%91%D0%B0%D0%B9%D0%BA%D0%B0%D0%BB%D1%8C%D1%...,nearby_feature_proxy,2.6,2.0,2.2
7,R026,Абзаково,%D0%9D%D0%BE%D0%B2%D0%BE%D0%B0%D0%B1%D0%B7%D0%...,nearby_feature_proxy,2.1,1.5,0.7
0,R002,Красная Поляна,%D0%9A%D1%80%D0%B0%D1%81%D0%BD%D0%B0%D1%8F_%D0...,resort_or_complex_page,2.0,0.0,2.0


In [6]:
from urllib.parse import unquote
disputed["страница_википедии"] = disputed["страница_википедии"].map(unquote)

In [7]:
ov = pd.read_csv("../data/reference/coordinate_overrides.csv", encoding="utf-8-sig").set_index("resort_id")
v2 = pd.DataFrame(json.load(open("../data/raw/resort_open_data_v2.json", encoding="utf-8"))["resorts"]).set_index("resort_id")

diff_km = pd.Series({i: haversine(ov.at[i, "lat_final"], ov.at[i, "lon_final"],
                                  v2.at[i, "latitude"], v2.at[i, "longitude"]) for i in ov.index})
changed = diff_km[diff_km > 0.01]
print("Точки, изменённые после сбора:", changed.round(2).to_dict() or "нет")

Точки, изменённые после сбора: нет


In [8]:
d = pd.read_csv("../data/prepared/coordinate_disputes.csv", encoding="utf-8-sig")
print(d.columns.tolist())
print(d.loc[d["решение"].isna() | d["обоснование"].isna(), "курорт"].tolist() if {"решение", "обоснование"} <= set(d.columns) else "нет колонок решение/обоснование")

['id', 'курорт', 'страница_википедии', 'качество_википедии', 'вики_от_ручной_км', 'вики_от_исходной_км', 'ручная_от_исходной_км']
нет колонок решение/обоснование


In [9]:
from pathlib import Path
path = Path("../data/reference/coordinate_disputes_decisions.csv")
if path.exists():
    print("Файл уже есть, не перезаписываю:", path)
else:
    d = pd.read_csv("../data/prepared/coordinate_disputes.csv", encoding="utf-8-sig")
    pd.DataFrame({"id": d["id"], "курорт": d["курорт"], "решение": "", "обоснование": ""}).to_csv(
        path, index=False, encoding="utf-8-sig")
    print("Создан шаблон:", path)

Файл уже есть, не перезаписываю: ../data/reference/coordinate_disputes_decisions.csv


In [10]:
pd.read_csv(path, encoding="utf-8-sig").columns.tolist()


['id', 'курорт', 'решение', 'обоснование']

In [11]:
dec = pd.read_csv(path, encoding="utf-8-sig", sep=None, engine="python")
dec.columns = [str(c).strip().lower() for c in dec.columns]
print("Колонки:", dec.columns.tolist())

need = {"курорт", "решение", "обоснование"}
assert need <= set(dec.columns), f"Нет колонок: {sorted(need - set(dec.columns))}"

dec = dec.fillna("")
print("Не заполнено:", dec.loc[(dec["решение"] == "") | (dec["обоснование"] == ""), "курорт"].tolist())

dec.to_csv(path, index=False, encoding="utf-8-sig")   # теперь файл в стандартном виде

Колонки: ['id', 'курорт', 'решение', 'обоснование']
Не заполнено: []


In [12]:
dec = pd.read_csv(path, encoding="utf-8-sig").fillna("")
print("Не заполнено:", dec.loc[(dec["решение"] == "") | (dec["обоснование"] == ""), "курорт"].tolist())

Не заполнено: []


In [13]:
import pandas as pd
from pathlib import Path

path = Path("../data/raw/prices_manual_2026.csv")

if path.exists():
    print("Файл уже есть, не перезаписываю (чтобы не стереть вашу работу):", path)
else:
    names = pd.read_csv("../data/reference/coordinate_overrides.csv", encoding="utf-8-sig") \
              .set_index("resort_id")["name_ru"].to_dict()

    need_ids = ["R030", "R006", "R029", "R004", "R016", "R019",
                "R028", "R018", "R008", "R020", "R026", "R021"]
    months = ["2026-12", "2027-01", "2027-02", "2027-03"]

    template = pd.DataFrame([
        {"resort_id": rid, "resort_name": names[rid], "month": m, "season": "2026/27",
         "tariff_product": "Ски-пасс на 1 день", "day_type": "", "age_category": "взрослый",
         "adult_price_min_rub": None, "adult_price_max_rub": None, "currency": "RUB",
         "source_quality": "", "source_url": "", "checked_at": "", "comment": ""}
        for rid in need_ids for m in months
    ])
    template.to_csv(path, index=False, encoding="utf-8-sig")
    print("Создан файл:", path, "| строк:", len(template))

Файл уже есть, не перезаписываю (чтобы не стереть вашу работу): ../data/raw/prices_manual_2026.csv


In [14]:
from pathlib import Path

path = Path("../data/raw/prices_manual_2026.csv")
px = pd.read_csv(path, encoding="utf-8-sig", sep=None, engine="python")
px.columns = [str(c).strip() for c in px.columns]
for c in ["resort_id", "tariff_product", "day_type", "pricing_model", "source_url", "source_quality"]:
    px[c] = px[c].fillna("").astype(str).str.strip()
px["comment"] = px["comment"].fillna("").astype(str)
px["source_url"] = px["source_url"].str.replace(r"\?(ysclid|utm_\w+)=.*$", "", regex=True)
px["lo"] = pd.to_numeric(px["adult_price_min_rub"], errors="coerce")
px["hi"] = pd.to_numeric(px["adult_price_max_rub"], errors="coerce")
px["hours"] = pd.to_numeric(px["duration_hours"], errors="coerce")
px["checked_dt"] = pd.to_datetime(px["checked_at"], errors="coerce", dayfirst=True)
px = px.drop_duplicates()

problems = []
def flag(mask, text):
    for _, r in px[mask].iterrows():
        problems.append((r["resort_id"], r["resort_name"], r["day_type"], text))

priced = px["pricing_model"].isin(["day_pass", "hourly"])
flag(~px["pricing_model"].isin(["day_pass", "hourly", "per_ride", "not_published"]), "pricing_model: day_pass, hourly, per_ride или not_published")
flag(priced & px["lo"].isna(), "нет минимальной цены")
flag(priced & px["hi"].isna() & ~px["comment"].str.contains(r"\bот\b", case=False), "нет максимальной цены (если цена «от», напишите это в комментарии)")
flag(priced & (px["lo"] > px["hi"]), "мин больше макс")
flag(priced & ((px["lo"] < 300) | (px["hi"] > 30000)), "цена вне диапазона 300–30000 ₽")
flag(~priced & (px["lo"].notna() | px["hi"].notna()), "для per_ride / not_published цены должны быть пустыми")
flag((px["pricing_model"] == "hourly") & ~(px["hours"] > 0), "для hourly нужна duration_hours")
flag((px["pricing_model"] == "day_pass") & px["hours"].notna(), "для day_pass duration_hours должна быть пустой")
flag(~px["source_url"].str.startswith("http"), "нет ссылки")
flag(~px["source_quality"].isin(["official", "secondary"]), "source_quality: official или secondary")
flag(px["checked_dt"].isna(), "нет даты проверки")
flag(px.duplicated(["resort_id", "tariff_product", "day_type"], keep=False), "дубль курорт+тариф+тип дня")
flag(px["comment"].str.contains("VIP", case=False) & ~px["comment"].str.contains("не включ", case=False) & priced,
     "в комментарии VIP: не вошла ли цена VIP-прохода в максимум?")

print(pd.DataFrame(problems, columns=["id", "курорт", "тип дня", "проблема"]).sort_values("id").to_string(index=False)
      if problems else "Ошибок не найдено")

base = px[(px["pricing_model"] == "day_pass") & px["lo"].notna()].copy()
base["пик"] = base["day_type"].str.contains("новогод", case=False)
avg = lambda d: pd.concat([d["lo"], d["hi"]]).mean()
mean_all = lambda d: pd.Series(d[["lo", "hi"]].to_numpy().ravel()).mean()
preview = pd.DataFrame({
    "с пиковыми строками": base.groupby("resort_name").apply(mean_all, include_groups=False),
    "без пиковых": base[~base["пик"]].groupby("resort_name").apply(mean_all, include_groups=False),
}).round(0)
print(preview.sort_values("без пиковых"))
print(px["pricing_model"].value_counts().to_dict(), "| курортов:", px["resort_id"].nunique())

Ошибок не найдено
                с пиковыми строками  без пиковых
resort_name                                     
Чегет                        1700.0       1700.0
Соболиная гора               2883.0       2525.0
Губаха                       2700.0       2700.0
Гора Белая                   2750.0       2750.0
Горный воздух                2800.0       2800.0
Шерегеш                      3125.0       3125.0
Абзаково                     3800.0       3800.0
{'day_pass': 12, 'hourly': 3, 'per_ride': 2} | курортов: 12
